# RAG-сервис генерации инструкций

Прототип сервиса, который по запросу пользователя ищет релевантные фрагменты
в базе знаний (портал госуслуг Санкт-Петербурга) и генерирует структурированную
инструкцию с помощью LLM.

# Пайплайн


In [4]:
import sys
from pathlib import Path

# Ноутбук лежит в notebook/, добавляем src/ в sys.path,
# чтобы импортировать модули пайплайна
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from logging_config import setup_logging
from retrieval import retrieve
from generation import generate
from schemas import Instruction

setup_logging()

In [5]:
def print_instruction(instr: Instruction) -> None:
    print("-" * 70)
    print(f"{instr.title}")
    print("-" * 70)
    print(f"Кому адресована: {instr.audience}")
    print(f"Описание: {instr.summary}")
    print()

    print("Необходимые документы:")
    if instr.required_documents:
        for doc in instr.required_documents:
            print(f"   - {doc}")
    else:
        print("   (не указаны)")
    print()

    print("Порядок действий:")
    for i, step in enumerate(instr.steps, start=1):
        print(f"   {i}. {step}")
    print()

    print(f"Сроки: {instr.deadline or 'не указаны'}")
    print(f"Куда обращаться: {instr.where_to_apply or 'не указано'}")
    print(f"Источников: {len(instr.sources)}")

In [6]:
query = "Как поступить в первый класс?"

chunks = retrieve(query, top_k=3)
print(f"Найдено чанков: {len(chunks)}")
for rc in chunks:
    print(f"  [{rc.score:.3f}] {rc.chunk.question}")

instruction = generate(query, chunks)
print()
print_instruction(instruction)

Найдено чанков: 3
  [0.843] Какие документы нужны?
  [0.840] Можно ли подавать заявление, если ребёнок младше 6,5 или старше 8 лет?
  [0.836] Как подать заявление?


03:15:03 [INFO] generation: Валидация прошла успешно



----------------------------------------------------------------------
Как подать заявление на поступление в первый класс
----------------------------------------------------------------------
Кому адресована: родители
Описание: Инструкция по подаче заявления на поступление в первый класс в Санкт-Петербурге.

Необходимые документы:
   - паспорт родителя (законного представителя)
   - свидетельство о рождении ребёнка
   - документ, подтверждающий проживание будущего первоклассника на закреплённой территории (постоянная или временная регистрация либо справка о приёме документов для оформления регистрации по месту жительства)
   - документ, подтверждающий законность нахождения ребёнка на территории России

Порядок действий:
   1. Соберите все необходимые документы: паспорт родителя (законного представителя), свидетельство о рождении ребёнка, документ, подтверждающий проживание будущего первоклассника на закреплённой территории, а также документ, подтверждающий законность нахождения ребён